# LC25000 colon_n — Partition-Refinement Pipeline

Refines the KMeans partition itself (not the representatives), targeting
**exactly 250 clusters** containing **exactly 5000 images** total.

Stages:
1. Setup, mount Drive, folder structure
2. Get raw colon_n images (restore from Drive backup, or download once)
3. Lab histograms + KMeans (k=250) + baseline variance metrics
4. Refinement pass — reassign members of small clusters to better fits,
   split oversized clusters and route pieces to matching clusters or new ones,
   ORB-verify uncertain assignments
5. Normalise to exactly 250 clusters (only if needed)
6. Post-refinement variance metrics + before/after comparison
7. Save one representative (sharpest image) per final cluster + manifest

**colon_aca is deferred** until this pipeline is validated on colon_n.

Every stage writes a `.done` marker to Drive. Reruns skip completed stages.
Every stage asserts total-image-count == 5000 as a hard invariant.

## Stage 1 — Setup

In [ ]:
!pip install -q datasets opencv-python-headless scikit-learn tqdm pillow pandas

In [ ]:
import os
import csv
import shutil

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.cluster import KMeans
from google.colab import drive

drive.mount("/content/drive")

# ---- Paths -------------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/LC25000_colon_v2"    # NEW root, won't touch old pipeline
RAW_BACKUP = f"{DRIVE_ROOT}/raw"
REP_DIR    = f"{DRIVE_ROOT}/representatives"
LOG_DIR    = f"{DRIVE_ROOT}/logs"
MARKER_DIR = f"{DRIVE_ROOT}/markers"

RAW_DIR = "/content/raw"          # local scratch, wiped on disconnect
RAW_ZIP = f"{RAW_BACKUP}/colon_raw_all.zip"

CLASS         = "colon_n"          # colon_aca deferred
N_CLUSTERS    = 250
N_IMAGES      = 5000
EXPECTED_SZ   = N_IMAGES / N_CLUSTERS   # 20

# ---- Refinement thresholds (tunable) -----------------------------------
HIST_BINS       = 32
SMALL_THRESH    = 0.5    # cluster is "small"    if size < 0.5 * expected  (< 10)
LARGE_THRESH    = 1.5    # cluster is "large"    if size > 1.5 * expected  (> 30)
ORB_MATCH_MIN   = 40     # ORB match count for two images to be "same original"
N_ORB_PROBES    = 3      # how many members of a target cluster to ORB-test against

# ---- Setup dirs --------------------------------------------------------
for d in [RAW_BACKUP, REP_DIR, f"{REP_DIR}/{CLASS}", LOG_DIR, MARKER_DIR]:
    os.makedirs(d, exist_ok=True)
os.makedirs(f"{RAW_DIR}/{CLASS}", exist_ok=True)


# ---- Stage markers -----------------------------------------------------
def marker_path(name): return os.path.join(MARKER_DIR, f"{name}.done")
def is_done(name):     return os.path.exists(marker_path(name))
def mark_done(name, note=""):
    with open(marker_path(name), "w") as f: f.write(note or "done")
def reset_stage(name):
    p = marker_path(name)
    if os.path.exists(p): os.remove(p); print(f"Reset: {name}")
    else: print(f"No marker for {name}")


# ---- Invariant check ---------------------------------------------------
def assert_total(labels, tag):
    n = len(labels)
    assert n == N_IMAGES, f"{tag}: expected {N_IMAGES} images, got {n}"
    print(f"  \u2713 {tag}: {n} images across {len(set(labels))} clusters")


print("\nReady. Completed stages:", sorted(f[:-5] for f in os.listdir(MARKER_DIR) if f.endswith(".done")) or "none")


Mounted at /content/drive

Ready. Completed stages: none


## Stage 2 — Get raw colon_n images

In [ ]:
def raw_present():
    return len(os.listdir(f"{RAW_DIR}/{CLASS}")) > 0

if raw_present():
    print("Raw images already on local scratch.")
elif os.path.exists(RAW_ZIP):
    print("Restoring from Drive backup...")
    shutil.unpack_archive(RAW_ZIP, RAW_DIR)
else:
    print("First run: downloading LC25000 from Hugging Face (~1.1GB)...")
    from datasets import load_dataset
    ds = load_dataset("1aurent/LC25000", split="train")
    organ_names = ds.features["organ"].names
    label_names = ds.features["label"].names
    colon_idx = organ_names.index("colon")
    benign_idx = label_names.index("benign")
    aca_idx = label_names.index("adenocarcinomas")

    os.makedirs(f"{RAW_DIR}/colon_aca", exist_ok=True)  # save both, back both up
    n_saved = {"colon_n": 0, "colon_aca": 0}
    for ex in tqdm(ds):
        if ex["organ"] != colon_idx: continue
        if ex["label"] == benign_idx: key = "colon_n"
        elif ex["label"] == aca_idx:  key = "colon_aca"
        else: continue
        ex["image"].save(os.path.join(f"{RAW_DIR}/{key}", f"{key}_{n_saved[key]:05d}.jpeg"))
        n_saved[key] += 1
    print("Downloaded:", n_saved)

    print("Backing up to Drive...")
    shutil.make_archive("/content/colon_raw_all", "zip", RAW_DIR)
    shutil.move("/content/colon_raw_all.zip", RAW_ZIP)

n_local = len(os.listdir(f"{RAW_DIR}/{CLASS}"))
assert n_local == N_IMAGES, f"Expected {N_IMAGES} raw images, got {n_local}"
print(f"\u2713 {n_local} raw {CLASS} images ready")
mark_done("stage2")


First run: downloading LC25000 from Hugging Face (~1.1GB)...


README.md:   0%|          | 0.00/1.71k [00:00<?, ?B/s]

data/train-00000-of-00004-40f80292828d4b(…): reconstructing file:   0%|          |  0.00B /  387MB            

data/train-00000-of-00004-40f80292828d4b(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-de395cdb9595cd(…): reconstructing file:   0%|          |  0.00B /  375MB            

data/train-00001-of-00004-de395cdb9595cd(…): downloading bytes:           |  0.00B            

... (3 outputs left out)

data/train-00003-of-00004-402d3c6cb6e672(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

100%|██████████| 25000/25000 [03:05<00:00, 135.01it/s]


Downloaded: {'colon_n': 5000, 'colon_aca': 5000}
Backing up to Drive...
✓ 5000 raw colon_n images ready


## Stage 3 — Lab histograms, KMeans (k=250), baseline variance

Two metrics computed on the histogram feature space:

- **Intra-cluster variance** — mean squared distance from each image in a
  cluster to that cluster's center. Low = images within a cluster look
  alike (good).
- **Inter-cluster variance** — mean squared distance between pairs of
  cluster centers. High = clusters are well separated from each other (good).

The ratio inter/intra is the standard *separation ratio* — higher is better
partition quality. We track this before and after refinement to prove the
refinement actually helped.

In [ ]:
def compute_lab_histogram(image_path, bins=HIST_BINS):
    img = cv2.imread(image_path)
    if img is None: return None
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)
    parts = []
    for ch in range(3):
        h = cv2.calcHist([lab], [ch], None, [bins], [0, 256])
        parts.append(cv2.normalize(h, h).flatten())
    return np.concatenate(parts)


def compute_variance_metrics(features, labels, centers=None):
    """Return dict with intra, inter, separation ratio."""
    if centers is None:
        centers = np.array([features[labels == k].mean(axis=0) if (labels == k).any()
                            else np.zeros(features.shape[1])
                            for k in range(labels.max() + 1)])
    # intra: mean squared distance to own center
    intra = np.mean([np.sum((features[i] - centers[labels[i]]) ** 2) for i in range(len(features))])
    # inter: mean squared pairwise distance between centers (non-empty only)
    non_empty = [k for k in range(len(centers)) if (labels == k).any()]
    if len(non_empty) < 2:
        inter = 0.0
    else:
        c = centers[non_empty]
        d2 = np.sum((c[:, None, :] - c[None, :, :]) ** 2, axis=2)
        iu = np.triu_indices(len(c), k=1)
        inter = float(d2[iu].mean())
    return {"intra": float(intra), "inter": float(inter),
            "ratio": float(inter / intra) if intra > 0 else 0.0,
            "n_clusters_nonempty": len(non_empty)}


files = sorted(f for f in os.listdir(f"{RAW_DIR}/{CLASS}") if f.lower().endswith(".jpeg"))
paths = [os.path.join(f"{RAW_DIR}/{CLASS}", f) for f in files]

if is_done("stage3"):
    print("Stage 3 already done \u2014 reloading features and labels")
    features = np.load(f"{LOG_DIR}/{CLASS}_features.npy")
    labels   = np.load(f"{LOG_DIR}/{CLASS}_labels_baseline.npy")
    centers  = np.load(f"{LOG_DIR}/{CLASS}_centers_baseline.npy")
else:
    print("Computing histograms...")
    feats = []
    for p in tqdm(paths):
        feats.append(compute_lab_histogram(p))
    features = np.array(feats)

    print(f"Running KMeans (k={N_CLUSTERS})...")
    km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
    labels = km.fit_predict(features)
    centers = km.cluster_centers_

    np.save(f"{LOG_DIR}/{CLASS}_features.npy", features)
    np.save(f"{LOG_DIR}/{CLASS}_labels_baseline.npy", labels)
    np.save(f"{LOG_DIR}/{CLASS}_centers_baseline.npy", centers)
    mark_done("stage3")

assert_total(labels, "Stage 3")

baseline_metrics = compute_variance_metrics(features, labels, centers)
sizes = pd.Series(labels).value_counts().sort_index()

print(f"\nBaseline partition ({CLASS}, k={N_CLUSTERS}):")
print(f"  Cluster sizes  : min {sizes.min()}, max {sizes.max()}, mean {sizes.mean():.1f}, std {sizes.std():.1f}")
print(f"  Empty clusters : {N_CLUSTERS - (sizes > 0).sum()}")
print(f"  Small (<{int(EXPECTED_SZ*SMALL_THRESH)}) : {(sizes < EXPECTED_SZ*SMALL_THRESH).sum()} clusters")
print(f"  Large (>{int(EXPECTED_SZ*LARGE_THRESH)}) : {(sizes > EXPECTED_SZ*LARGE_THRESH).sum()} clusters")
print(f"\n  Intra-cluster variance : {baseline_metrics['intra']:.6f}")
print(f"  Inter-cluster variance : {baseline_metrics['inter']:.6f}")
print(f"  Separation ratio       : {baseline_metrics['ratio']:.3f}   (higher = better)")

pd.DataFrame([baseline_metrics]).to_csv(f"{LOG_DIR}/{CLASS}_metrics_baseline.csv", index=False)


Computing histograms...


100%|██████████| 5000/5000 [01:11<00:00, 70.18it/s]


Running KMeans (k=250)...
  ✓ Stage 3: 5000 images across 250 clusters

Baseline partition (colon_n, k=250):
  Cluster sizes  : min 4, max 67, mean 20.0, std 9.0
  Empty clusters : 0
  Small (<10) : 14 clusters
  Large (>30) : 25 clusters

  Intra-cluster variance : 0.006261
  Inter-cluster variance : 0.478631
  Separation ratio       : 76.447   (higher = better)


## Stage 4 — Refinement

Two operations, both operate on *membership* (which cluster each image
belongs to), not on representatives.

**Small-cluster dissolution** — for each cluster below the small threshold,
each of its members is reassigned to whichever other cluster's *center* it
is closest to in histogram space. The original small cluster becomes empty.

**Large-cluster splitting + rerouting** — for each cluster above the large
threshold, run local KMeans to break it into ~n/expected pieces. For each
piece:

1. Find the closest existing cluster by histogram distance to its center.
2. Confirm with ORB: sample a few images from the target cluster, ORB-match
   against the piece's own sharpest image. If enough matches score above
   `ORB_MATCH_MIN`, merge the piece in. Otherwise keep it as a new cluster.

ORB descriptors are cached so we don't recompute per comparison.

In [ ]:
# ORB helpers with caching
_ORB = cv2.ORB_create(nfeatures=500)
_DESC_CACHE = {}

def orb_descriptors(path):
    if path not in _DESC_CACHE:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        _, des = _ORB.detectAndCompute(img, None) if img is not None else (None, None)
        _DESC_CACHE[path] = des
    return _DESC_CACHE[path]

_BF = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

def orb_match_count(path_a, path_b, dist_thresh=40):
    da, db = orb_descriptors(path_a), orb_descriptors(path_b)
    if da is None or db is None: return 0
    return sum(1 for m in _BF.match(da, db) if m.distance < dist_thresh)


def sharpness_score(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: return 0.0
    return float(cv2.Laplacian(img, cv2.CV_64F).var())


def cluster_center(features, labels, cid):
    mask = labels == cid
    return features[mask].mean(axis=0) if mask.any() else None


def dissolve_small_cluster(features, labels, cid, live_centers, live_ids):
    """Reassign each member of cluster `cid` to the closest live cluster."""
    members = np.where(labels == cid)[0]
    for idx in members:
        # squared distance to each live center
        d2 = np.sum((live_centers - features[idx]) ** 2, axis=1)
        # exclude self
        for i, lid in enumerate(live_ids):
            if lid == cid: d2[i] = np.inf
        labels[idx] = live_ids[int(np.argmin(d2))]
    return labels


def split_large_cluster(features, paths, labels, cid, next_new_id):
    """Split cluster cid into n_sub pieces. Route each piece to a matching existing
    cluster (histogram + ORB confirm) or promote it to a new cluster ID."""
    member_idx = np.where(labels == cid)[0]
    if len(member_idx) < 2: return labels, next_new_id, []

    n_sub = int(round(len(member_idx) / EXPECTED_SZ))
    n_sub = max(2, min(n_sub, len(member_idx)))
    sub_feats = features[member_idx]
    sub_km = KMeans(n_clusters=n_sub, random_state=42, n_init=10)
    sub_labels = sub_km.fit_predict(sub_feats)

    routes = []   # (piece_size, action, target_id, orb_hits)

    live_ids = [k for k in range(labels.max() + 1)
                if (labels == k).any() and k != cid]
    live_centers = np.array([cluster_center(features, labels, k) for k in live_ids])

    for sid in range(n_sub):
        piece_local = np.where(sub_labels == sid)[0]
        piece_global = member_idx[piece_local]
        if len(piece_global) == 0: continue

        piece_center = sub_feats[piece_local].mean(axis=0)

        # histogram: which live cluster's center is closest?
        d2 = np.sum((live_centers - piece_center) ** 2, axis=1)
        best_target = live_ids[int(np.argmin(d2))]

        # ORB confirm: piece's sharpest image vs. a few target images
        piece_sharp = piece_global[int(np.argmax([sharpness_score(paths[i]) for i in piece_global]))]
        target_members = np.where(labels == best_target)[0]
        probes = target_members[:N_ORB_PROBES] if len(target_members) <= N_ORB_PROBES \
                 else np.random.RandomState(cid + sid).choice(target_members, N_ORB_PROBES, replace=False)
        orb_hits = [orb_match_count(paths[piece_sharp], paths[p]) for p in probes]
        max_hits = max(orb_hits) if orb_hits else 0

        if max_hits >= ORB_MATCH_MIN:
            labels[piece_global] = best_target
            routes.append((int(len(piece_global)), "merged", int(best_target), int(max_hits)))
        else:
            labels[piece_global] = next_new_id
            routes.append((int(len(piece_global)), "new", int(next_new_id), int(max_hits)))
            next_new_id += 1

    return labels, next_new_id, routes


if is_done("stage4"):
    print("Stage 4 already done \u2014 reloading refined labels")
    labels_refined = np.load(f"{LOG_DIR}/{CLASS}_labels_refined.npy")
else:
    labels_refined = labels.copy()
    audit = []   # rows for the reassignment log

    sizes_now = pd.Series(labels_refined).value_counts()
    small_ids = sorted(sizes_now[sizes_now < EXPECTED_SZ * SMALL_THRESH].index.tolist())
    large_ids = sorted(sizes_now[sizes_now > EXPECTED_SZ * LARGE_THRESH].index.tolist())

    print(f"Refining: {len(small_ids)} small clusters to dissolve, "
          f"{len(large_ids)} large clusters to split")

    # ---- dissolve small clusters ----
    print("\nDissolving small clusters...")
    for cid in tqdm(small_ids):
        n_before = int((labels_refined == cid).sum())
        if n_before == 0: continue
        live_ids = [k for k in np.unique(labels_refined) if k != cid]
        live_centers = np.array([cluster_center(features, labels_refined, k) for k in live_ids])
        labels_refined = dissolve_small_cluster(features, labels_refined, cid, live_centers, live_ids)
        audit.append({"op": "dissolve", "orig_cluster": int(cid), "n_moved": n_before,
                      "target": "nearest", "orb_hits": None})

    # ---- split large clusters ----
    print("\nSplitting large clusters...")
    next_new_id = int(labels_refined.max() + 1)
    for cid in tqdm(large_ids):
        labels_refined, next_new_id, routes = split_large_cluster(
            features, paths, labels_refined, cid, next_new_id)
        for size, action, target, hits in routes:
            audit.append({"op": f"split_{action}", "orig_cluster": int(cid),
                          "n_moved": size, "target": target, "orb_hits": hits})

    pd.DataFrame(audit).to_csv(f"{LOG_DIR}/{CLASS}_refinement_audit.csv", index=False)
    np.save(f"{LOG_DIR}/{CLASS}_labels_refined.npy", labels_refined)
    mark_done("stage4")

assert_total(labels_refined, "Stage 4")

sizes_after = pd.Series(labels_refined).value_counts()
print(f"\nAfter refinement:")
print(f"  Distinct non-empty cluster IDs: {len(sizes_after)}")
print(f"  Sizes: min {sizes_after.min()}, max {sizes_after.max()}, "
      f"mean {sizes_after.mean():.1f}, std {sizes_after.std():.1f}")


Refining: 14 small clusters to dissolve, 25 large clusters to split

Dissolving small clusters...


100%|██████████| 14/14 [00:00<00:00, 27.68it/s]



Splitting large clusters...


100%|██████████| 25/25 [00:19<00:00,  1.26it/s]

  ✓ Stage 4: 5000 images across 256 clusters

After refinement:
  Distinct non-empty cluster IDs: 256
  Sizes: min 8, max 65, mean 19.5, std 6.6


## Stage 5 — Normalise to exactly 250 clusters

After refinement, the number of non-empty clusters may be slightly above or
below 250. This stage nudges the count to exactly 250 by the smallest
intervention possible:

- If **above 250**: repeatedly merge the smallest cluster into its
  histogram-nearest neighbor.
- If **below 250**: repeatedly split the largest cluster into two using
  local KMeans.

Also **relabels** clusters to be a contiguous range 0..249, so downstream
code can treat cluster IDs cleanly.

In [ ]:
if is_done("stage5"):
    print("Stage 5 already done \u2014 reloading final labels")
    labels_final = np.load(f"{LOG_DIR}/{CLASS}_labels_final.npy")
else:
    labels_work = labels_refined.copy()

    def unique_ids(lbl):
        return sorted(np.unique(lbl).tolist())

    non_empty = unique_ids(labels_work)
    n_current = len(non_empty)
    print(f"Starting normalization: {n_current} non-empty clusters, target {N_CLUSTERS}")

    # ---- If too many: merge smallest into nearest ----
    while n_current > N_CLUSTERS:
        non_empty = unique_ids(labels_work)
        sizes = {k: int((labels_work == k).sum()) for k in non_empty}
        smallest = min(sizes, key=sizes.get)

        centers = {k: cluster_center(features, labels_work, k) for k in non_empty}
        c_small = centers[smallest]
        d2 = {k: float(np.sum((centers[k] - c_small) ** 2)) for k in non_empty if k != smallest}
        nearest = min(d2, key=d2.get)

        labels_work[labels_work == smallest] = nearest
        n_current -= 1

    # ---- If too few: split largest ----
    next_new_id = int(labels_work.max() + 1)
    while n_current < N_CLUSTERS:
        non_empty = unique_ids(labels_work)
        sizes = {k: int((labels_work == k).sum()) for k in non_empty}
        largest = max(sizes, key=sizes.get)

        member_idx = np.where(labels_work == largest)[0]
        if len(member_idx) < 2: break
        sub = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(features[member_idx])
        labels_work[member_idx[sub == 1]] = next_new_id
        next_new_id += 1
        n_current += 1

    # ---- Relabel to contiguous 0..249 ----
    remap = {old: new for new, old in enumerate(unique_ids(labels_work))}
    labels_final = np.array([remap[l] for l in labels_work])

    np.save(f"{LOG_DIR}/{CLASS}_labels_final.npy", labels_final)
    mark_done("stage5")

# ---- Hard invariants ----
sizes_final = pd.Series(labels_final).value_counts()
n_distinct = len(sizes_final)
n_total = int(sizes_final.sum())
assert n_distinct == N_CLUSTERS, f"Expected exactly {N_CLUSTERS} clusters, got {n_distinct}"
assert n_total == N_IMAGES, f"Expected exactly {N_IMAGES} images, got {n_total}"
print(f"\u2713 Exactly {n_distinct} clusters")
print(f"\u2713 Exactly {n_total} images total")
print(f"\nFinal cluster sizes: min {sizes_final.min()}, max {sizes_final.max()}, "
      f"mean {sizes_final.mean():.1f}, std {sizes_final.std():.1f}")


Starting normalization: 256 non-empty clusters, target 250
✓ Exactly 250 clusters
✓ Exactly 5000 images total

Final cluster sizes: min 10, max 65, mean 20.0, std 6.6


## Stage 6 — Variance metrics: baseline vs. final

The whole point of the refinement was to improve partition quality. This
stage recomputes intra and inter variance on the *final* labels and
compares to baseline. Successful refinement moves the separation ratio up.

In [ ]:
final_centers = np.array([features[labels_final == k].mean(axis=0)
                          for k in range(N_CLUSTERS)])
final_metrics = compute_variance_metrics(features, labels_final, final_centers)

print("Variance metrics — baseline vs. final:\n")
print(f"{'':22s}  {'baseline':>10s}  {'final':>10s}  {'change':>12s}")
for k in ["intra", "inter", "ratio"]:
    b, f_ = baseline_metrics[k], final_metrics[k]
    if k == "intra":
        change = f"{(f_ - b) / b * 100:+.1f}% (want down)"
    else:
        change = f"{(f_ - b) / b * 100:+.1f}% (want up)"
    print(f"{k:22s}  {b:>10.6f}  {f_:>10.6f}  {change:>12s}")

pd.DataFrame([{**baseline_metrics, "stage": "baseline"},
              {**final_metrics, "stage": "final"}]).to_csv(
    f"{LOG_DIR}/{CLASS}_metrics_comparison.csv", index=False)


Variance metrics — baseline vs. final:

                          baseline       final        change
intra                     0.006261    0.006805  +8.7% (want down)
inter                     0.478631    0.439449  -8.2% (want up)
ratio                    76.446846   64.581665  -15.5% (want up)


## Stage 7 — Save one representative per final cluster + manifest

Sharpest image (highest Laplacian variance) per final cluster. Written to
Drive, along with a manifest listing every representative with its cluster
size and source path.

In [ ]:
rep_rows = []
print("Saving representatives...")
for cid in tqdm(range(N_CLUSTERS)):
    member_idx = np.where(labels_final == cid)[0]
    member_paths = [paths[i] for i in member_idx]
    rep = member_paths[int(np.argmax([sharpness_score(p) for p in member_paths]))]
    out_name = f"cluster{cid:04d}_{os.path.basename(rep)}"
    shutil.copy(rep, os.path.join(f"{REP_DIR}/{CLASS}", out_name))
    rep_rows.append({"cluster_id": cid, "n_members": len(member_paths),
                     "representative_src": rep, "saved_as": out_name})

manifest_path = f"{LOG_DIR}/{CLASS}_final_manifest.csv"
pd.DataFrame(rep_rows).to_csv(manifest_path, index=False)
mark_done("stage7")

on_disk = len([f for f in os.listdir(f"{REP_DIR}/{CLASS}") if f.lower().endswith(".jpeg")])
print(f"\n\u2713 {on_disk} representatives on Drive")
print(f"\u2713 Manifest: {manifest_path}")
print(f"\nColon_n pipeline complete.")
print(f"Drive folder to copy into QuPath's images/: {REP_DIR}/{CLASS}")


Saving representatives...


 10%|█         | 25/250 [00:04<00:36,  6.19it/s]


KeyboardInterrupt: 

---
## Notes

- **To rerun a stage**: `reset_stage("stage3")` etc., then re-execute that
  cell. Later stages that depend on it will re-derive from its outputs on
  their next run.
- **All state on Drive** (features, labels at each stage, audit log, metrics
  comparison, manifest). Nothing important lives only on Colab's local disk.
- **colon_aca**: once this is validated on colon_n, change `CLASS = "colon_n"`
  at the top of Stage 1 to `CLASS = "colon_aca"`, delete every `.done`
  marker, and re-run. The Drive layout (`representatives/`, `logs/`) keeps
  per-class files separate automatically.